### Building a RAG System with LangChain and ChromaDB
#### Introduction
Retrieval-Augmented Generation (RAG) is a powerful technique that combines the capabilities of large language models with external knowledge retrieval. This notebook will walk you through building a complete RAG system using:

- LangChain: A framework for developing applications powered by language models
- ChromaDB: An open-source vector database for storing and retrieving embeddings
- OpenAI: For embeddings and language model (you can substitute with other providers)

In [1]:
import os
# to use our OPENAI API key
from dotenv import load_dotenv 
load_dotenv()

True

In [2]:
# langchain imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
# from langchain.schema import Document - older legacy

# vector store imports
# from langchain_community.vectorstores import Chroma - legacy
from langchain_chroma import Chroma

# utility imports
import numpy as np
from typing import List



In [3]:
# RAG Architecture Overview
print("""
RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge
""")


RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge



### 1. Sample Data

In [4]:
## create sample documents
sample_docs = [
    """
    Machine Learning Fundamentals
    
    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised 
    learning finds patterns in unlabeled data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties.
    """,
    
    """
    Deep Learning and Neural Networks
    
    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of interconnected 
    nodes. Deep learning has revolutionized fields like computer vision, natural language 
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly 
    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers 
    excel at sequential data processing.
    """,
    
    """
    Natural Language Processing (NLP)
    
    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, 
    machine translation, and question answering. Modern NLP heavily relies on transformer 
    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand 
    context and relationships between words in text.
    """
]

sample_docs

['\n    Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.\n    ',
 '\n    Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective f

In [5]:
## save sample documents to files
import tempfile
temp_dir=tempfile.mkdtemp()

for i,doc in enumerate(sample_docs):
    with open(f"{temp_dir}/doc_{i}.txt","w") as f:
        f.write(doc)

print(f"Sample document create in : {temp_dir}")

Sample document create in : C:\Users\itsme\AppData\Local\Temp\tmptz_9i1sn


In [6]:
## save sample documents to files
import os

# 1. Store the path string first
target_dir = "data/" 

# 2. Create the directory (ignores return value)
os.makedirs(target_dir, exist_ok=True) 

# 3. Use the string variable to build file paths
for i, doc in enumerate(sample_docs):
    with open(f"{target_dir}/doc_{i}.txt", "w") as f:
        f.write(doc)

print(f"Sample documents created in: {target_dir}")


Sample documents created in: data/


### 2. Document Loading

In [7]:
from langchain_community.document_loaders import DirectoryLoader , TextLoader

# Load Documents from Directory 
loader = DirectoryLoader(
    "data",
    loader_cls=TextLoader,
    glob="*.txt",
    loader_kwargs={'encoding' : 'utf-8'}

)

documents = loader.load()

print(f"Loaded {len(documents)} documents")
print(f"\n First Document Preview")
print(documents[0].page_content)


Loaded 3 documents

 First Document Preview

    Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised 
    learning finds patterns in unlabeled data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties.
    


In [8]:
documents

[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='\n    Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.\n    '),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='\n    Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natur

### 3. Document Splitting

In [9]:
# Initialize text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,  # Maximum size of each chunk
    chunk_overlap=50,  # Overlap between chunks to maintain context
    length_function=len,
    separators=[" "]  # Hierarchy of separators
)
chunks=text_splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks from {len(documents)} documents")
print(f"\nChunk example:")
print(f"Content: {chunks[0].page_content[:150]}...")
print(f"Metadata: {chunks[0].metadata}")

Created 5 chunks from 3 documents

Chunk example:
Content: Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experie...
Metadata: {'source': 'data\\doc_0.txt'}


In [10]:
chunks

[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
 Document(metadata={'source': 'data\\doc_0.txt'}, page_content='data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.'),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of in

### 4. Embedding Models

In [11]:
os.environ['OPENAI_ENV_KEY'] = os.getenv('OPENAI_API_KEY')

In [12]:
sample_text= "ML is Wonderful , and I love building ML models"
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x000001F5A9A2BA10>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x000001F5AAF24440>, model='text-embedding-3-large', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [13]:
vector = embeddings.embed_query(sample_text)
print(vector)
print(len(vector))

[0.004154205322265625, 0.0107269287109375, -0.016998291015625, -0.006801605224609375, 0.0404052734375, 0.01444244384765625, -0.029998779296875, -0.0119476318359375, 0.02667236328125, 0.03228759765625, 0.0211334228515625, -0.018890380859375, 0.0031375885009765625, -0.0034637451171875, -0.004833221435546875, -0.0037078857421875, 0.0123291015625, -0.01483917236328125, -0.01849365234375, -0.0185089111328125, -0.0236968994140625, -0.052154541015625, -0.032958984375, 0.0245513916015625, -0.00865936279296875, -0.0011606216430664062, -0.0074005126953125, 0.01087188720703125, -0.0242919921875, 0.0024204254150390625, 0.034393310546875, 0.005596160888671875, 0.025970458984375, 0.0175018310546875, 0.0016088485717773438, -0.0283203125, 0.01629638671875, 0.02862548828125, 0.00885009765625, 0.02923583984375, 0.00226593017578125, 0.004177093505859375, -0.005718231201171875, 0.0220794677734375, 0.02105712890625, -0.019317626953125, -0.0031108856201171875, -0.0180816650390625, 0.0034313201904296875, 0.0

### Intilialize the ChromaDB Vector Store And Stores the chunks in Vector Representation

In [14]:
chunks

[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
 Document(metadata={'source': 'data\\doc_0.txt'}, page_content='data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.'),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of in

In [15]:
# Create a cromadb vector store
persist_directory = "./chroma_db"

# Initialize chromadb with openAI embeddings
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=OpenAIEmbeddings(model="text-embedding-3-small"),
    persist_directory=persist_directory,
    collection_name="rag_collection"
)

print(f" Vector store created with {vectorstore._collection.count()} vectors")
print(f"Persisted to : {persist_directory}")

 Vector store created with 10 vectors
Persisted to : ./chroma_db


### 5. Test Similarity Search

In [22]:
query = "What are the types of machine learning?"

similar_docs = vectorstore.similarity_search(query , k=3)
similar_docs


[Document(id='501c3c87-febe-42a1-8b58-9eddca0d4393', metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
 Document(id='74780762-2d30-459e-a799-b07acad6191f', metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and 

In [18]:
query = "What is NLP?"

similar_docs = vectorstore.similarity_search(query , k=3)
similar_docs

[Document(id='7abc94cc-bfa8-4677-8609-a8895dcba731', metadata={'source': 'data\\doc_2.txt'}, page_content='Natural Language Processing (NLP)\n\n    NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, \n    machine translation, and question answering. Modern NLP heavily relies on transformer \n    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand \n    context and relationships between words in text.'),
 Document(id='b1896922-bd5e-4b50-a773-58823fd253f3', metadata={'source': 'data\\doc_2.txt'}, page_content='Natural Language Processing (NLP)\n\n    NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, \n    machine translation, and question answering. Modern NLP heavily relies on 

In [19]:
query = "What is DL?"

similar_docs = vectorstore.similarity_search(query , k=3)
similar_docs

[Document(id='866cbaa9-a59e-4396-96ff-a326073e324c', metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers'),
 Document(id='b51f6c65-5e51-49de-a717-fe050e2803eb', metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer 

In [20]:
print(f"Query: {query}")
print(f"\nTop {len(similar_docs)} similar chunks:")
for i, doc in enumerate(similar_docs):
    print(f"\n--- Chunk {i+1} ---")
    print(doc.page_content[:200] + "...")
    print(f"Source: {doc.metadata.get('source', 'Unknown')}")

Query: What is DL?

Top 3 similar chunks:

--- Chunk 1 ---
Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of i...
Source: data\doc_1.txt

--- Chunk 2 ---
Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of i...
Source: data\doc_1.txt

--- Chunk 3 ---
data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties....
Source: data\doc_0.txt


### 6. Advanced Similarity Search With Scores

In [23]:
result_score = vectorstore.similarity_search_with_score(query , k=3)
result_score

[(Document(id='501c3c87-febe-42a1-8b58-9eddca0d4393', metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
  0.7797614336013794),
 (Document(id='74780762-2d30-459e-a799-b07acad6191f', metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, un

#### Understanding Similarity Scores
The similarity score represents how closely related a document chunk is to your query. The scoring depends on the distance metric used:

ChromaDB default: Uses L2 distance (Euclidean distance)

- Lower scores = MORE similar (closer in vector space)
- Score of 0 = identical vectors
- Typical range: 0 to 2 (but can be higher)


Cosine similarity (if configured):

- Higher scores = MORE similar
- Range: -1 to 1 (1 being identical)

#### 7. Initialize LLM, RAG Chain, Prompt Template,Query the RAG system

In [26]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini-2024-07-18",
    temperature=0.2,
    max_tokens=500
)
# use max_complted_tokens for newer reasoning models

In [27]:
test_response = llm.invoke("What is LLM")
test_response

AIMessage(content='LLM stands for "Large Language Model." It refers to a type of artificial intelligence model that is designed to understand and generate human-like text based on the input it receives. These models are typically trained on vast amounts of text data and use deep learning techniques, particularly neural networks, to learn patterns in language.\n\nKey characteristics of LLMs include:\n\n1. **Scale**: They are "large" in terms of the number of parameters (often billions or even trillions), which allows them to capture complex language patterns.\n\n2. **Versatility**: LLMs can perform a wide range of language-related tasks, including text generation, translation, summarization, question answering, and more.\n\n3. **Contextual Understanding**: They can generate contextually relevant responses by considering the input text and the broader context in which it appears.\n\n4. **Transfer Learning**: Many LLMs are pre-trained on a diverse dataset and can be fine-tuned for specifi

In [28]:
from langchain.chat_models.base import init_chat_model

llm = init_chat_model("openai:gpt-4o-mini-2024-07-18")
# llm = init_chat_model("groq:gjajja.....2028..... version") -- similar for any other models
llm

ChatOpenAI(output_version=None, client=<openai.resources.chat.completions.completions.Completions object at 0x000001F5AB401CD0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001F5AB402650>, root_client=<openai.OpenAI object at 0x000001F5AB5B8950>, root_async_client=<openai.AsyncOpenAI object at 0x000001F5AB5B8830>, model_name='gpt-4o-mini-2024-07-18', model_kwargs={}, openai_api_key=SecretStr('**********'), openai_proxy=None, stream_usage=True)

In [29]:
llm.invoke("What is AI")

AIMessage(content='Artificial Intelligence (AI) refers to the simulation of human intelligence in machines programmed to think and learn like humans. This field of computer science involves creating algorithms and systems that can perform tasks typically requiring human intelligence, such as understanding natural language, recognizing patterns, solving problems, and making decisions.\n\nAI can be categorized into two main types:\n\n1. **Narrow AI (Weak AI)**: This type of AI is designed and trained for specific tasks. Examples include virtual assistants like Siri or Alexa, image recognition systems, and recommendation algorithms used by streaming services.\n\n2. **General AI (Strong AI)**: This is a theoretical form of AI that possesses the ability to understand, learn, and apply intelligence in a way comparable to a human being, capable of performing a wide range of cognitive tasks. As of now, General AI does not exist.\n\nAI technologies include machine learning, deep learning, natur

### 8. Modern RAG Chain

In [31]:
from langchain.chains import create_retrieval_chain
# above import is depreceated but functionality is- Creates a pipeline from the context we are getting to the large language model to generate a response (output). This is the final step in the RAG architecture.
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain

ModuleNotFoundError: No module named 'langchain.chains'

In [35]:
# Option 1: The Modern LangChain Way (Recommended)
# Instead of relying on rigid, pre-built functional wrappers that get shifted across packages, the standard best practice is to declare the pipeline explicitly using LCEL (LangChain Expression Language) via the | (pipe) operator.

# It completely removes the need for create_retrieval_chain, keeps your code lightning-fast, and stops version changes from breaking your imports.

# Here we are using Option 2: The Quick Patch (Using the Classic Module)

In [33]:
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

In [34]:
## Convert vector store to retriever
retriever=vectorstore.as_retriever(
    search_kwarg={"k":3} ## Retrieve top 3 relevant chunks
)
retriever
# We need to convert our vector store to a retriever so that we can use it in the RAG pipeline ( we can include our vector store retrievel chain). The retriever will handle the retrieval of relevant documents based on the query provided by the user.
# Vector Store Retriever is just like a interface to the vectore store. It allows us to use the vector store in a more abstract way, focusing on the retrieval of relevant documents without worrying about the underlying implementation details of the vector store itself.

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000001F5AAF24980>, search_kwargs={})